<a href="https://colab.research.google.com/github/sensein/asd-ai-scoping-review/blob/main/scripts/PRISMA_pipeline_Fabio/pubmed_id_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests biopython

In [ ]:
# Mount Google Drive
from google.colab import drive as gdrive
import os
import csv
import pandas as pd
import pandas as pd
from collections import OrderedDict
import re

In [ ]:
gdrive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
workspace = "/content/drive/MyDrive/[ICON fellowship] social robot and autism/AI for behavioral analysis and autism/queries/PubMed"

In [ ]:
# Folder containing .bib file
pubmed_folder = f"{workspace}/PubMed/"

# Output CSV file
output_csv = f"{workspace}/autism.csv"

output_csv_formatted = f"{workspace}/autism_formatted.csv"

In [ ]:
# Get a list of all Pubmed files in the folder
pubmed_files = [file for file in os.listdir(pubmed_folder) if file.endswith('.txt')]

In [ ]:
ids = []
# Check if there are any Pubmed files in the folder
if len(pubmed_files) == 0:
    print("No CSV files found in the folder.")
else:
    # Loop through the Pubmed files and merge them into the DataFrame
    for pubmed_file in pubmed_files:
        file_path = os.path.join(pubmed_folder, pubmed_file)
        print(file_path)

        with open(file_path, 'r') as file:
            text = file.read()

        # Split the text into individual entries using "PMID-" as a delimiter
        entries = re.split(r'\nPMID-', text)[1:]

        # Loop through the entries and extract information
        for entry in entries:
            lines = entry.strip().split('\n')
            pmid_line = lines[0].strip()  # Get the PMID line
            ids.append(pmid_line)
ids = list(set(ids))

/content/drive/MyDrive/[ICON fellowship] social robot and autism/AI for behavioral analysis and autism/queries/PubMed/PubMed/pubmed-autismTitl-set.txt
/content/drive/MyDrive/[ICON fellowship] social robot and autism/AI for behavioral analysis and autism/queries/PubMed/PubMed/pubmed-autismTitl-set (1).txt
/content/drive/MyDrive/[ICON fellowship] social robot and autism/AI for behavioral analysis and autism/queries/PubMed/PubMed/pubmed-autismTitl-set (2).txt
/content/drive/MyDrive/[ICON fellowship] social robot and autism/AI for behavioral analysis and autism/queries/PubMed/PubMed/pubmed-autismTitl-set (3).txt
/content/drive/MyDrive/[ICON fellowship] social robot and autism/AI for behavioral analysis and autism/queries/PubMed/PubMed/pubmed-autismTitl-set (4).txt
/content/drive/MyDrive/[ICON fellowship] social robot and autism/AI for behavioral analysis and autism/queries/PubMed/PubMed/pubmed-autismTitl-set (5).txt
/content/drive/MyDrive/[ICON fellowship] social robot and autism/AI for be

In [ ]:
len(ids)

48296

In [ ]:
if os.path.exists(f"{workspace}/autism.csv"):
    all_df = pd.read_csv(f"{workspace}/autism.csv")
else:
    all_df = pd.DataFrame([], columns=["PMID", "Title", "Abstract", "Keywords", "DOI", "Link", "Authors", "Journal", "Year"])

In [ ]:
existing_ids = list(all_df["PMID"])
existing_ids = [str(id) for id in existing_ids]
len(existing_ids)

48125

In [ ]:
# Use a list comprehension to find IDs not in existing_ids
ids = [str(id) for id in ids if str(id) not in existing_ids]
print(len(ids))

171


In [ ]:
from Bio import Entrez
# Replace with your PubMed API key
API_KEY = os.getenv("NCBI_API_KEY")

Entrez.email = os.getenv("NCBI_EMAIL")
Entrez.api_key = API_KEY
# Join the list of PubMed IDs into a comma-separated string
id_list = ",".join(ids)
handle = Entrez.efetch(db="pubmed", id=id_list, retmode="xml")
records = Entrez.read(handle)
my_list = []

for record in records['PubmedArticle']:
  print("HELLO")
  try:
    title = record['MedlineCitation']['Article']['ArticleTitle']
  except:
    title = None

  try:
    abstract = record['MedlineCitation']['Article']['Abstract']['AbstractText']
    abstract = ''.join(abstract)
  except:
    abstract = None

  try:
    keywords = record['MedlineCitation']['KeywordList']
    keywords = [str(item) for item in keywords[0]]
  except:
    keywords = None

  try:
    authors = [author['LastName'] for author in record['MedlineCitation']['Article']['AuthorList']]
    authors = ', '.join(authors)
  except:
    authors = None

  try:
    journal = record['MedlineCitation']['Article']['Journal']['Title']
  except:
    journal = None

  try:
    year = record['MedlineCitation']['Article']['Journal']['JournalIssue']['PubDate']['Year']
  except:
    year = None

  try:
    for element in record['PubmedData']['ArticleIdList']:
      id_type = element.attributes.get('IdType')
      id_value = str(element)

      if id_type == 'pubmed':
        pubmed_id = id_value
      elif id_type == 'doi':
        doi = id_value

    link = f"https://pubmed.ncbi.nlm.nih.gov/{pubmed_id}/"
  except:
    link = None
    doi = None
  print(pubmed_id)
  my_list.append([pubmed_id, title, abstract, keywords, doi, link, authors, journal, year])

  # Save the DataFrame every 100 new fetched abstracts
  if len(my_list) % 10 == 0:
    my_df = pd.DataFrame(my_list, columns=["PMID", "Title", "Abstract", "Keywords", "DOI", "Link", "Authors", "Journal", "Year"])
    all_df = pd.concat([all_df, my_df], ignore_index=True)
    all_df.to_csv(f"{workspace}/autism.csv", index=False)
    my_list = []
    print(len(all_df))

my_df = pd.DataFrame(my_list, columns=["PMID", "Title", "Abstract", "Keywords", "DOI", "Link", "Authors", "Journal", "Year"])
all_df = pd.concat([all_df, my_df], ignore_index=True)
all_df.to_csv(f"{workspace}/autism.csv", index=False)
handle.close()

In [ ]:
all_df = pd.read_csv(f"{workspace}/autism.csv")
# Define a function to join the lists of strings
def join_keywords(keyword_list):
  if isinstance(keyword_list, list):
    return ', '.join(keyword_list)
  else:
    return keyword_list


# Apply the function to the 'Keywords' column
all_df['Keywords'] = all_df['Keywords'].apply(join_keywords)
all_df = all_df.dropna(subset=['Abstract'])
all_df

,PMID,Title,Abstract,Keywords,DOI,Link,Authors,Journal,Year
0,35333369,Decision flexibilities in autism spectrum diso...,People make flexible decisions across a wide r...,"['autism spectrum disorder', 'flexibility', 'f...",10.1093/scan/nsac023,https://pubmed.ncbi.nlm.nih.gov/35333369/,"Tei, Tanicha, Itahashi, Aoki, Ohta, Qian, Hash...",Social cognitive and affective neuroscience,2022.0
1,35102227,Alternative female and male developmental traj...,"The numerous multistable phenomena in vision, ...",NaN,10.1038/s41598-022-05620-1,https://pubmed.ncbi.nlm.nih.gov/35102227/,"Ziman, Aleshin, Unoka, Braun, Kovács",Scientific reports,2022.0
2,32604886,"Reelin Functions, Mechanisms of Action and Sig...","During embryonic development and adulthood, Re...","['Reelin', 'cellular pathways', 'cerebral cort...",10.3390/biom10060964,https://pubmed.ncbi.nlm.nih.gov/32604886/,Jossin,Biomolecules,2020.0
4,36618123,<i>FMR1</i> gene CGG repeat distribution among...,Fragile X syndrome is the most common genetic ...,"['CGG repeat variation', 'autism spectrum diso...",10.1002/ggn2.10048,https://pubmed.ncbi.nlm.nih.gov/36618123/,"Nagarathinam, Chong, B K, Justin Margret, Venk...","Advanced genetics (Hoboken, N.J.)",2021.0
5,23543291,Brief report: atypical neuromagnetic responses...,Atypical auditory perception is a widely recog...,NaN,10.1007/s10803-013-1805-z,https://pubmed.ncbi.nlm.nih.gov/23543291/,"Brock, Bzishvili, Reid, Hautus, Johnson",Journal of autism and developmental disorders,2013.0
...,...,...,...,...,...,...,...,...,...
48120,27478377,ESSENCE-Q - a first clinical validation study ...,Early identification of autism spectrum disord...,"['ESSENCE', 'ESSENCE-Q', 'cutoff levels', 'rec...",10.2147/NDT.S108411,https://pubmed.ncbi.nlm.nih.gov/27478377/,"Hatakenaka, Fernell, Sakaguchi, Ninomiya, Fuku...",Neuropsychiatric disease and treatment,2016.0
48121,25290267,Elevated 5-hydroxymethylcytosine in the Engrai...,Epigenetic mechanisms regulate programmed gene...,NaN,10.1038/tp.2014.87,https://pubmed.ncbi.nlm.nih.gov/25290267/,"James, Shpyleva, Melnyk, Pavliv, Pogribny",Translational psychiatry,2014.0
48122,36336205,In Context: A Developmental Model of Reward Pr...,Differences in reward processing have been ass...,"['autism', 'infancy', 'motivation', 'reward']",10.1016/j.jaac.2022.07.861,https://pubmed.ncbi.nlm.nih.gov/36336205/,"Clements, Ascunce, Nelson",Journal of the American Academy of Child and A...,2022.0
48123,25070471,Change in autism symptoms and maladaptive beha...,Little is known about outcomes for individuals...,NaN,10.1007/s10803-014-2199-2,https://pubmed.ncbi.nlm.nih.gov/25070471/,"Woodman, Smith, Greenberg, Mailick",Journal of autism and developmental disorders,2015.0


In [ ]:
all_df = pd.read_csv(f"{workspace}/autism.csv")
all_df = all_df.drop('PMID', axis=1)
all_df.to_csv(f"{workspace}/autism.csv", index=False)
all_df

,Title,Abstract,Keywords,DOI,Link,Authors,Journal,Year
0,Decision flexibilities in autism spectrum diso...,People make flexible decisions across a wide r...,"['autism spectrum disorder', 'flexibility', 'f...",10.1093/scan/nsac023,https://pubmed.ncbi.nlm.nih.gov/35333369/,"Tei, Tanicha, Itahashi, Aoki, Ohta, Qian, Hash...",Social cognitive and affective neuroscience,2022.0
1,Alternative female and male developmental traj...,"The numerous multistable phenomena in vision, ...",NaN,10.1038/s41598-022-05620-1,https://pubmed.ncbi.nlm.nih.gov/35102227/,"Ziman, Aleshin, Unoka, Braun, Kovács",Scientific reports,2022.0
2,"Reelin Functions, Mechanisms of Action and Sig...","During embryonic development and adulthood, Re...","['Reelin', 'cellular pathways', 'cerebral cort...",10.3390/biom10060964,https://pubmed.ncbi.nlm.nih.gov/32604886/,Jossin,Biomolecules,2020.0
3,Transcranial magnetic stimulation (TMS) therap...,NaN,"['Autism Spectrum Disorder', 'consensus', 'pat...",10.3389/fnhum.2014.01034,https://pubmed.ncbi.nlm.nih.gov/25642178/,"Oberman, Enticott, Casanova, Rotenberg, Pascua...",Frontiers in human neuroscience,2014.0
4,<i>FMR1</i> gene CGG repeat distribution among...,Fragile X syndrome is the most common genetic ...,"['CGG repeat variation', 'autism spectrum diso...",10.1002/ggn2.10048,https://pubmed.ncbi.nlm.nih.gov/36618123/,"Nagarathinam, Chong, B K, Justin Margret, Venk...","Advanced genetics (Hoboken, N.J.)",2021.0
...,...,...,...,...,...,...,...,...
48120,ESSENCE-Q - a first clinical validation study ...,Early identification of autism spectrum disord...,"['ESSENCE', 'ESSENCE-Q', 'cutoff levels', 'rec...",10.2147/NDT.S108411,https://pubmed.ncbi.nlm.nih.gov/27478377/,"Hatakenaka, Fernell, Sakaguchi, Ninomiya, Fuku...",Neuropsychiatric disease and treatment,2016.0
48121,Elevated 5-hydroxymethylcytosine in the Engrai...,Epigenetic mechanisms regulate programmed gene...,NaN,10.1038/tp.2014.87,https://pubmed.ncbi.nlm.nih.gov/25290267/,"James, Shpyleva, Melnyk, Pavliv, Pogribny",Translational psychiatry,2014.0
48122,In Context: A Developmental Model of Reward Pr...,Differences in reward processing have been ass...,"['autism', 'infancy', 'motivation', 'reward']",10.1016/j.jaac.2022.07.861,https://pubmed.ncbi.nlm.nih.gov/36336205/,"Clements, Ascunce, Nelson",Journal of the American Academy of Child and A...,2022.0
48123,Change in autism symptoms and maladaptive beha...,Little is known about outcomes for individuals...,NaN,10.1007/s10803-014-2199-2,https://pubmed.ncbi.nlm.nih.gov/25070471/,"Woodman, Smith, Greenberg, Mailick",Journal of autism and developmental disorders,2015.0


In [ ]:
all_df = pd.read_csv(output_csv_formatted)
all_df


,Title,Abstract,Keywords,DOI,Link,Authors,Journal,Year
0,Decision flexibilities in autism spectrum diso...,People make flexible decisions across a wide r...,"['autism spectrum disorder', 'flexibility', 'f...",10.1093/scan/nsac023,https://pubmed.ncbi.nlm.nih.gov/35333369/,"Tei, Tanicha, Itahashi, Aoki, Ohta, Qian, Hash...",Social cognitive and affective neuroscience,2022.0
1,Alternative female and male developmental traj...,"The numerous multistable phenomena in vision, ...",NaN,10.1038/s41598-022-05620-1,https://pubmed.ncbi.nlm.nih.gov/35102227/,"Ziman, Aleshin, Unoka, Braun, Kovács",Scientific reports,2022.0
2,"Reelin Functions, Mechanisms of Action and Sig...","During embryonic development and adulthood, Re...","['Reelin', 'cellular pathways', 'cerebral cort...",10.3390/biom10060964,https://pubmed.ncbi.nlm.nih.gov/32604886/,Jossin,Biomolecules,2020.0
3,Transcranial magnetic stimulation (TMS) therap...,NaN,"['Autism Spectrum Disorder', 'consensus', 'pat...",10.3389/fnhum.2014.01034,https://pubmed.ncbi.nlm.nih.gov/25642178/,"Oberman, Enticott, Casanova, Rotenberg, Pascua...",Frontiers in human neuroscience,2014.0
4,<i>FMR1</i> gene CGG repeat distribution among...,Fragile X syndrome is the most common genetic ...,"['CGG repeat variation', 'autism spectrum diso...",10.1002/ggn2.10048,https://pubmed.ncbi.nlm.nih.gov/36618123/,"Nagarathinam, Chong, B K, Justin Margret, Venk...","Advanced genetics (Hoboken, N.J.)",2021.0
...,...,...,...,...,...,...,...,...
48120,ESSENCE-Q - a first clinical validation study ...,Early identification of autism spectrum disord...,"['ESSENCE', 'ESSENCE-Q', 'cutoff levels', 'rec...",10.2147/NDT.S108411,https://pubmed.ncbi.nlm.nih.gov/27478377/,"Hatakenaka, Fernell, Sakaguchi, Ninomiya, Fuku...",Neuropsychiatric disease and treatment,2016.0
48121,Elevated 5-hydroxymethylcytosine in the Engrai...,Epigenetic mechanisms regulate programmed gene...,NaN,10.1038/tp.2014.87,https://pubmed.ncbi.nlm.nih.gov/25290267/,"James, Shpyleva, Melnyk, Pavliv, Pogribny",Translational psychiatry,2014.0
48122,In Context: A Developmental Model of Reward Pr...,Differences in reward processing have been ass...,"['autism', 'infancy', 'motivation', 'reward']",10.1016/j.jaac.2022.07.861,https://pubmed.ncbi.nlm.nih.gov/36336205/,"Clements, Ascunce, Nelson",Journal of the American Academy of Child and A...,2022.0
48123,Change in autism symptoms and maladaptive beha...,Little is known about outcomes for individuals...,NaN,10.1007/s10803-014-2199-2,https://pubmed.ncbi.nlm.nih.gov/25070471/,"Woodman, Smith, Greenberg, Mailick",Journal of autism and developmental disorders,2015.0


In [ ]:
# Define a dictionary to map the old column names to the new ones
column_mapping = OrderedDict([
    ('Title', 'Title'),
    ('Abstract', 'Abstract'),
    ('Keywords', 'Keywords'),
    ('DOI', 'DOI'),
    ('Link', 'URL'),
    ('Authors', 'Authors'),
    ('Journal', 'Venue'),
    ('Year', 'Year')
])

column_order = ['Title', 'Abstract', 'Keywords', 'DOI', 'URL', 'Authors', 'Venue', 'Year']

# Rename the columns based on the mapping
all_df.rename(columns=column_mapping, inplace=True)

# List of columns to drop
columns_to_drop = [col for col in all_df.columns if col not in column_mapping.values()]

# Drop the unwanted columns
all_df.drop(columns=columns_to_drop, inplace=True)

all_df = all_df[column_order]
all_df

,Title,Abstract,Keywords,DOI,URL,Authors,Venue,Year
0,Decision flexibilities in autism spectrum diso...,People make flexible decisions across a wide r...,"['autism spectrum disorder', 'flexibility', 'f...",10.1093/scan/nsac023,https://pubmed.ncbi.nlm.nih.gov/35333369/,"Tei, Tanicha, Itahashi, Aoki, Ohta, Qian, Hash...",Social cognitive and affective neuroscience,2022.0
1,Alternative female and male developmental traj...,"The numerous multistable phenomena in vision, ...",NaN,10.1038/s41598-022-05620-1,https://pubmed.ncbi.nlm.nih.gov/35102227/,"Ziman, Aleshin, Unoka, Braun, Kovács",Scientific reports,2022.0
2,"Reelin Functions, Mechanisms of Action and Sig...","During embryonic development and adulthood, Re...","['Reelin', 'cellular pathways', 'cerebral cort...",10.3390/biom10060964,https://pubmed.ncbi.nlm.nih.gov/32604886/,Jossin,Biomolecules,2020.0
3,Transcranial magnetic stimulation (TMS) therap...,NaN,"['Autism Spectrum Disorder', 'consensus', 'pat...",10.3389/fnhum.2014.01034,https://pubmed.ncbi.nlm.nih.gov/25642178/,"Oberman, Enticott, Casanova, Rotenberg, Pascua...",Frontiers in human neuroscience,2014.0
4,<i>FMR1</i> gene CGG repeat distribution among...,Fragile X syndrome is the most common genetic ...,"['CGG repeat variation', 'autism spectrum diso...",10.1002/ggn2.10048,https://pubmed.ncbi.nlm.nih.gov/36618123/,"Nagarathinam, Chong, B K, Justin Margret, Venk...","Advanced genetics (Hoboken, N.J.)",2021.0
...,...,...,...,...,...,...,...,...
48120,ESSENCE-Q - a first clinical validation study ...,Early identification of autism spectrum disord...,"['ESSENCE', 'ESSENCE-Q', 'cutoff levels', 'rec...",10.2147/NDT.S108411,https://pubmed.ncbi.nlm.nih.gov/27478377/,"Hatakenaka, Fernell, Sakaguchi, Ninomiya, Fuku...",Neuropsychiatric disease and treatment,2016.0
48121,Elevated 5-hydroxymethylcytosine in the Engrai...,Epigenetic mechanisms regulate programmed gene...,NaN,10.1038/tp.2014.87,https://pubmed.ncbi.nlm.nih.gov/25290267/,"James, Shpyleva, Melnyk, Pavliv, Pogribny",Translational psychiatry,2014.0
48122,In Context: A Developmental Model of Reward Pr...,Differences in reward processing have been ass...,"['autism', 'infancy', 'motivation', 'reward']",10.1016/j.jaac.2022.07.861,https://pubmed.ncbi.nlm.nih.gov/36336205/,"Clements, Ascunce, Nelson",Journal of the American Academy of Child and A...,2022.0
48123,Change in autism symptoms and maladaptive beha...,Little is known about outcomes for individuals...,NaN,10.1007/s10803-014-2199-2,https://pubmed.ncbi.nlm.nih.gov/25070471/,"Woodman, Smith, Greenberg, Mailick",Journal of autism and developmental disorders,2015.0


In [ ]:
all_df.to_csv(output_csv_formatted, index=False)